# Soft Computing Assignment – 2

# Motivation & Idea
According to Jamali et al. [1], few algorithms are inspired by plant behaviour. In fact, many algorithms seem to be inspired by animals. This piqued our interest in developing a plant-based algorithm rather than an animal-based one.
Using information from [2], we adapted an algorithm that specifically mimics the growth of plant roots. We also discovered that several forces influence the direction of root growth.
The main ones are hydrotropism, the tendency to grow towards water resources, and gravitropism, the general tendency for roots to grow downwards. Unlike animals, which move towards visible targets, plant roots navigate blindly through soil, continuously adapting their growth in response to invisible environmental gradients. Their ability to explore, compete and optimise without centralised control provides a compelling model for designing robust optimisation algorithms.

### References
- [1]: Jamali et al(2025): A Systematic Review of Bio-Inspired Metaheuristic Optimization Algorithms: The Untapped Potential of Plant-Based Approaches 
- [2]: Hodge et al(2009): Plant root growth, architecture and function

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.frontiersin.org%2Ffiles%2FArticles%2F1120583%2Ffpls-14-1120583-HTML-r1%2Fimage_m%2Ffpls-14-1120583-g001.jpg&f=1&nofb=1&ipt=9aa9c5732380295e1a45d7c84d4d9e22609925c30afd312e70ed91400f0747ed" alt="Description" width="300" height="200">

## Algorithm

```txt
Each root agent has:

position
energy
growth direction
personal best

At each iteration:

Each root senses:
local fitness-derived moisture gradient
gravity bias
local competition repulsion
random exploratory perturbation
It grows one step in the normalized weighted sum direction.
It evaluates the new position:
if fitness improved, energy increases
otherwise energy decreases by growth cost
If energy exceeds a threshold:
the root branches into children
children inherit partial energy and slightly diverging directions
If energy falls to zero:
the root dies
Roots update the shared moisture field:
promising regions become wetter
occupied regions may be depleted
```

---


### Pseudo Code

```py
Initialize dynamic moisture field M
Initialize a few root agents at random positions
For each root:
    evaluate objective
    set personal best
    set initial energy

while at least one root is alive:
    new_roots = []

    for each alive root:
        sense local moisture bias
        sense repulsion from nearby roots
        sample random exploration direction
        combine influences into growth direction

        propose new position
        apply boundary handling
        evaluate objective at new position

        improvement = old_fitness - new_fitness
        energy += reward_from(improvement) - movement_cost

        if improvement > 0:
            deposit moisture around new position

        deplete moisture at current position
        update personal best

        if energy <= 0:
            mark dead
        elif energy >= split_threshold and population < max_root_count:
            create child roots in nearby perturbed directions
            share energy between parent and children
            append children to new_roots

    add new_roots to population

return best personal best among all roots
```

# Mathematical Formulization

## 1. Optimization problem

Let the objective be

$$
\min_{x \in D} f(x),
$$

where the feasible domain is the box

$$
D = \prod_{j=1}^d [\ell_j, u_j] \subset \mathbb{r}^d.
$$

Each root agent lives in $ \mathbb{r}^d $, with position constrained by projection onto $D$. 

## 2. Root population state

At iteration (t), root (i) has state

$$
R_i(t) = \Big(x_i(t), e_i(t),, p_i(t),, f_i(t),, s_i(t)\Big),
$$

where:

* $x_i(t) \in D$: current position,
* $e_i(t) \ge 0$: energy,
* $p_i(t)$: personal best position,
* $f_i(t) = f(x_i(t))$: current fitness,
* $s_i(t) \in \{{\text{ALIVE}, \text{DEAD}}\}$. 

The personal best satisfies

$$
p_i(t) = \arg\min_{\tau \le t} f(x_i(\tau)).
$$


## 3. Moisture field

The environment is a dynamic scalar field $M(t)(x)$, represented as a Gaussian kernel sum plus a baseline:

$$
M(t)(x)
=

M_0 + \sum_{k=1}^{K_t} a_k(t)
\exp!\left(
-\frac{|x-c_k(t)|^2}{2\sigma^2}
\right),
$$

where:

* $M_0$ > 0$: base moisture,
* $c_k(t)$: center of patch (k),
* $a_k(t)$: strength of patch (k) (positive = attractive, negative = depleted),
* $\sigma > 0$: kernel width. 

The gradient used for hydrotropism is

$$
\nabla M(t)(x)
=

\sum_{k=1}^{K_t}
a_k(t)
\exp!\left(
-\frac{|x-c_k(t)|^2}{2\sigma^2}
\right)
\left(
-\frac{x-c_k(t)}{\sigma^2}
\right).
$$

This is exactly the Gaussian gradient implemented in the code. 

Patch strengths evaporate over time:

$$
a_k(t+1) = (1-\rho), a_k(t),
$$

where $\rho \in (0,1)$ is the evaporation rate, and patches with very small magnitude are removed. 


## 4. Direction components

Each living root computes a movement direction from five normalized components.

Define the normalization operator

$$
\operatorname{norm}(v)=
\begin{cases}
\dfrac{v}{|v|}, & |v|>\varepsilon,[4pt]
0, & \text{otherwise}.
\end{cases}
$$



For root (i), the components are:

### Hydrotropism

$$
h_i(t) = \operatorname{norm}!\big(\nabla M(t)(x_i(t))\big).
$$

### Gravity

Given gravity vector (g),
$$
g^\sharp = \operatorname{norm}(g).
$$

### Random exploration

$$
r_i(t) = \operatorname{norm}(\xi_i(t)), \qquad \xi_i(t) \sim \mathcal{N}(0,I_d).
$$

### Competition repulsion

For repulsion radius (R), the repulsion from other alive roots is

$$
q_i(t)
=

\sum_{j\ne i}
\mathbf{1}{0<|x_i(t)-x_j(t)|<R}
\left(1-\frac{|x_i(t)-x_j(t)|}{R}\right)
\frac{x_i(t)-x_j(t)}{|x_i(t)-x_j(t)|},
$$

then normalized as

$$
c_i(t) = \operatorname{norm}(q_i(t)).
$$

### Personal-best attraction

$$
b_i(t) = \operatorname{norm}(p_i(t) - x_i(t)).
$$

These are the exact ingredients used in `calculate_direction`. 


## 5. Movement rule

Let the weights be

$$
(w_h, w_g, w_r, w_c)
$$

for hydrotropism, gravity, randomness, and competition, and let (w_b) be the personal-best weight.

Then the unnormalized direction is

$$
v_i(t)
=

w_h h_i(t)

* w_g g^\sharp
* w_r r_i(t)
* w_c c_i(t)
* w_b b_i(t).
$$

If $v_i(t)$ is numerically zero, a random Gaussian direction is substituted. The final direction is

$$
d_i(t) = \operatorname{norm}(v_i(t)).
$$



The adaptive step length is

$$
\eta_i(t)
=

\Delta \cdot \operatorname{clip}!\left(\frac{e_i(t)}{5},,0.25,,2.0\right),
$$

where $\Delta$ is the base step size. 

The proposed new position is

$$
\tilde{x}_i(t+1) = x_i(t) + \eta_i(t) d_i(t),
$$

followed by box projection:

$$
x_i(t+1) = \Pi_D!\left(\tilde{x}_i(t+1)\right).
$$


## 6. Fitness improvement and energy update

After moving, the new fitness is

$$
f_i(t+1) = f(x_i(t+1)).
$$

Improvement is defined for minimization as

$$
\Delta f_i(t) = f_i(t) - f_i(t+1).
$$

The algorithm uses **relative improvement**

$$
\gamma_i(t)
=

\operatorname{clip}!\left(
\frac{\max(\Delta f_i(t),0)}{|f_i(t)|+\varepsilon},
0,1
\right).
$$

Then energy updates according to

$$
e_i(t+1)
=

e_i(t) + \alpha \gamma_i(t) - c_{\text{step}},
$$

where $\alpha$ is the reward factor and $c_{\text{step}}$ is the movement cost. If $e_i(t+1) \le 0$, the root dies. 

This is one of the central mechanisms of the optimizer: good moves replenish energy, while poor moves gradually kill agents.


## 7. Personal-best update

If the new position improves the root’s historical best, then

$$
\text{if } f_i(t+1) < f(p_i(t)), \quad
p_i(t+1) = x_i(t+1),
$$

otherwise

$$
p_i(t+1) = p_i(t).
$$


## 8. Moisture deposition and depletion

The algorithm modifies the field based on root behavior.

### Positive deposit from successful movement

If $\gamma_i(t) > 0$, add a positive moisture patch at $x_i(t+1)$ with strength

$$
a_{\text{dep}} = \beta_{\text{dep}} , \gamma_i(t),
$$

where $\beta_{\text{dep}}$ is `deposit_scale`. 

### Extra deposit for new personal best

If a new personal best is found, add another deposit of size

$$
a_{\text{best}} = 0.5,\beta_{\text{dep}}.
$$


### Depletion from repeated visitation

Every move adds a negative patch at the current location:

$$
a_{\text{deplete}} = -\beta_{\text{depl}},
$$

where $\beta_{\text{depl}}$ is `depletion_scale`. 

So the field learns which regions are promising, but also discourages overcrowding and repeated exploitation.


## 9. Splitting rule

If a root has enough energy,

$$
e_i(t) \ge e_{\text{split}},
$$

and population capacity allows it, then it can split into children. 

Suppose $m_i$ children are created. Let $\lambda \in (0,1)$ be the split ratio. Then:

* total child energy:
  $$
  e_{\text{child,total}} = \lambda e_i(t),
  $$

* each child receives
  $$
  e_{\text{child}} = \frac{\lambda e_i(t)}{m_i},
  $$

* the parent keeps
  $$
  e_i(t+1) = (1-\lambda)e_i(t).
  $$



Each child is placed near the parent:

$$
x_{i,k}(t)
=

\Pi_D!\left(
x_i(t) + \delta, \hat{u}_{i,k}(t)
\right),
$$

where $\delta$ is `child_offset`, and

$$
\hat{u}_{i,k}(t)
=

\operatorname{norm}!\left(
\hat{d}*i(t) + 0.3,\zeta*{i,k}(t)
\right),
\qquad
\zeta_{i,k}(t) \sim \mathcal{N}(0,I_d),
$$

with $\hat{d}_i(t)$ being the parent’s last movement direction. 

This means children inherit the parent’s search tendency with random perturbation.

## 10. Compact formulation of one iteration

A compact iteration for each living root (i) is:

$$
d_i(t) =
\operatorname{norm}\Big(
w_h h_i(t) + w_g g^\sharp + w_r r_i(t) + w_c c_i(t) + w_b b_i(t)
\Big),
$$

$$
x_i(t+1) =
\Pi_D!\left(
x_i(t) + \Delta \cdot \operatorname{clip}!\left(\frac{e_i(t)}{5},0.25,2\right)d_i(t)
\right),
$$

$$
\gamma_i(t)=
\operatorname{clip}!\left(
\frac{\max(f(x_i(t))-f(x_i(t+1)),0)}{|f(x_i(t))|+\varepsilon},
0,1
\right),
$$

$$
e_i(t+1) = e_i(t) + \alpha \gamma_i(t) - c_{\text{step}}.
$$

Then:

* deposit positive moisture if $\gamma_i(t) > 0$,
* deposit bonus moisture if personal best improves,
* always deplete the visited region,
* kill root if $e_i(t+1) \le 0$. 


# The "Anti-Derivative" Proof

| Algorithm/Feature | PSO              | ACO           | GA               | PlantAlgorithm                   |
|-------------------|------------------|---------------|------------------|----------------------------------|
| Search Space      | Continuous       | Discrete      | Both             | Continuous                       |
| Memory            | Best Position    | Pheromone     | Implicit         | moisture field (Landscape-based) |
| Communication     | Global best      | Pheromone     | Selection        | moisture field (Landscape-based) |
| Update Rule       | Point attraction | Probabilistic | Genetic ops      | Field gradient  + forces         |
| Population        | Fixed            | Fixed         | Fixed            | Adaptive (spilt & death)         |
| Dynamics          | Velocity-based   | Path-based    | Generation-based | field-energy feedback system     |


Although the proposed algorithm belongs to the class of swarm-based metaheuristics, its underlying search dynamics differ fundamentally from classical approaches such as Particle Swarm Optimization (PSO), Ant Colony Optimization (ACO), and Genetic Algorithms (GA).

**Difference from PSO:**
PSO is fundamentally an attraction-based, velocity-driven system, where particles iteratively adjust their velocities toward their personal best and the global best positions. This results in a strong convergence bias toward known optima.

In contrast, the proposed algorithm does not rely on velocity updates or direct attraction to a global best. Instead, movement is governed by a combination of environmental gradient following and local repulsion mechanisms. Agents respond to a spatially distributed fitness field rather than explicitly chasing a best-performing agent. This shifts the paradigm from agent-centric attraction to environment-driven adaptation, resulting in more distributed and less centralized search behavior.

**Difference from ACO:**
ACO relies on indirect communication via pheromone trails, where agents reinforce paths that lead to better solutions. The collective intelligence emerges through the accumulation and evaporation of pheromone signals over time.

The proposed algorithm does not use any form of shared scalar memory such as pheromones. Instead, agents interact through local spatial competition and environmental feedback, without maintaining a global communication medium. Information is not accumulated globally but is instead embedded implicitly in the fitness landscape and the spatial distribution of agents. This eliminates the reliance on historical path reinforcement and introduces a more reactive, real-time adaptation mechanism.

**Difference from GA:**
Genetic Algorithms operate through population evolution mechanisms, including selection, crossover, and mutation. New candidate solutions are generated by recombining existing ones, and weaker individuals are replaced over generations.

In contrast, the proposed method does not use reproduction or recombination. Agents persist and evolve through continuous spatial movement, rather than discrete generational updates. The concept of "fitness improvement" is achieved through growth and directional adaptation, not genetic inheritance. This makes the algorithm inherently continuous and spatially grounded, rather than population-replacement-based.

# Expolation vs Exploitation

## Exploration
- **Random Direction**
Random Direction Noise added each step, roots off predictable paths.
- **Competition Repulsion**
Nearby roots push each other away.
- **Root Splitting**
Branch with  random offset.
- **Moisture Depletion**
Revisited areas lose water, let the roots reach into unexplored areas.

Exploration is primarily driven by stochastic growth and environmental uncertainty. Each agent (root tip) incorporates a random perturbation component in its movement, allowing it to probe previously unexplored regions of the search space. This randomness simulates the natural irregularity in root growth caused by soil resistance and micro-environmental variability. Additionally, the initialization of multiple agents across the domain further enhances global coverage, reducing the likelihood of premature convergence.

Another important exploration mechanism is the competition-induced repulsion between nearby roots. When multiple agents occupy similar regions, a repulsive interaction discourages overcrowding, forcing some agents to diverge toward less explored areas. This behavior prevents clustering around suboptimal regions and promotes spatial diversity in the population.

## Exploitation
- **Hydrotropism**
Roots follow moisture gradient toward areas with the fastest increase in moisture.
- **Personal Best** 
Roots are attracted to their own personal best.
- **Adaptive Step Size**
Low-energy roots take smaller steps; high-energy roots take larger steps.
- **Moisture Deposition**
Finding a good solution attracts other roots to move closer.

Exploitation is achieved through directional growth toward favorable environmental gradients, analogous to hydrotropism in plant roots. Agents bias their movement toward regions with higher fitness values (interpreted as higher moisture or resource availability), enabling convergence toward promising solutions.

Furthermore, the algorithm incorporates a memory component, where agents retain information about previously encountered high-quality positions. This allows them to refine their search locally and stabilize around optima once discovered.

## Balancing Mechanism
The balance between exploration and exploitation is governed by the relative influence of stochastic perturbation, environmental gradient following, and inter-agent competition. Early in the optimization process, random movement and spatial dispersion dominate, encouraging broad exploration. As the search progresses and high-quality regions are identified, gradient-following and memory-based adjustments become more influential, leading to intensified local search.

This adaptive interplay ensures that the algorithm avoids premature convergence while still achieving efficient refinement near global optima.

## Hyperparameter Optimization Test

The developed algorithm is stochastic and has a few parameters to adjust that can influence the result of the algorithm.
Below is a randomized grid search setup. It samples parameter combinations, runs multiple seeds on the 4 benchmark functions, and ranks configurations by mean normalized score.

We noticed that changing weights can have significance on the performance of the algorithm. Instrestingly, Overreliance on the moisture gradient results in bad performance possibly, because the roots tend to get stuck more often in local minima. Weighing in the other factors does seem to benefit the algorithm.

Overall, the hyperparameter optimization reveals that the proposed algorithm performs best under a regime of controlled growth, moderate energy availability, and strong environmental guidance. The results highlight that performance is not driven by a single parameter, but by the interaction between branching dynamics, repulsion, and gradient-following behavior. In particular, the importance of environmental weights over memory-based terms supports the claim that the algorithm operates as a field-driven adaptive system, rather than a classical swarm-based attraction model.

In [ ]:
from __future__ import annotations

from RootAlgorithm import PlantAlgorithm
from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple, Any, Sequence
import csv
import json
import math
import random
import numpy as np


# ------------------------------------------------------------
# Benchmark functions
# ------------------------------------------------------------

def rastrigin(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    n = x.size
    return 10.0 * n + np.sum(x * x - 10.0 * np.cos(2.0 * np.pi * x))


def ackley(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    n = x.size
    s1 = np.sum(x * x)
    s2 = np.sum(np.cos(2.0 * np.pi * x))
    return (
        -20.0 * np.exp(-0.2 * np.sqrt(s1 / n))
        - np.exp(s2 / n)
        + 20.0
        + math.e
    )


def schwefel(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    n = x.size
    return 418.9829 * n - np.sum(x * np.sin(np.sqrt(np.abs(x))))


def rosenbrock(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return np.sum(100.0 * (x[1:] - x[:-1] ** 2) ** 2 + (1.0 - x[:-1]) ** 2)


# ------------------------------------------------------------
# Benchmark problem definitions
# ------------------------------------------------------------

@dataclass
class BenchmarkProblem:
    name: str
    func: Callable[[np.ndarray], float]
    bounds: Sequence[Tuple[float, float]]
    optimum: float = 0.0


def make_problems(dimension: int = 3) -> List[BenchmarkProblem]:
    return [
        BenchmarkProblem("Rastrigin", rastrigin, [(-5.12, 5.12)] * dimension, 0.0),
        BenchmarkProblem("Ackley", ackley, [(-32.768, 32.768)] * dimension, 0.0),
        BenchmarkProblem("Schwefel", schwefel, [(-500.0, 500.0)] * dimension, 0.0),
        BenchmarkProblem("Rosenbrock", rosenbrock, [(-5.0, 10.0)] * dimension, 0.0),
    ]


# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------

def compute_step_size(bounds, scale=0.05):
    bounds_arr = np.asarray(bounds, dtype=float)
    widths = bounds_arr[:, 1] - bounds_arr[:, 0]
    return scale * float(np.mean(widths))


def sample_param_config(
    rng: random.Random,
    search_space: Dict[str, Sequence[Any]],
) -> Dict[str, Any]:
    return {k: rng.choice(list(v)) for k, v in search_space.items()}


def stable_normalized_score(
    value: float,
    optimum: float = 0.0,
    eps: float = 1e-12,
) -> float:
    return abs(value - optimum) / max(abs(optimum), 1.0, eps)


def mean_std(values: Sequence[float]) -> Tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    return float(np.mean(arr)), float(np.std(arr))


def serialize_value(v: Any) -> Any:
    if isinstance(v, np.ndarray):
        return v.tolist()
    if isinstance(v, tuple):
        return list(v)
    return v


# ------------------------------------------------------------
# Core evaluation
# ------------------------------------------------------------

def evaluate_config(
    config: Dict[str, Any],
    problems: Sequence[BenchmarkProblem],
    seeds: Sequence[int],
    max_iterations: int = 1200,
    step_scale: float = 0.05,
) -> Dict[str, Any]:
    """
    Requires PlantAlgorithm to already be defined/imported.
    step_size is computed separately for each problem from its bounds.
    """
    per_problem: Dict[str, Dict[str, Any]] = {}
    aggregate_scores: List[float] = []

    for problem in problems:
        fitnesses: List[float] = []
        positions: List[np.ndarray] = []

        problem_step_size = compute_step_size(problem.bounds, scale=step_scale)

        for seed in seeds:
            algo = PlantAlgorithm(
                objective_function=problem.func,
                bounds=problem.bounds,
                max_root_count=config["max_root_count"],
                initial_energy=config["initial_energy"],
                step_size=problem_step_size,
                step_cost=config["step_cost"],
                splitting_threshold=config["splitting_threshold"],
                max_children=config["max_children"],
                alpha=config["alpha"],
                deposit_scale=config["deposit_scale"],
                depletion_scale=config["depletion_scale"],
                weights=config["weights"],
                repulsion_radius=config["repulsion_radius"],
                gravity_vector=np.array(config["gravity_vector"], dtype=float),
                personal_best_weight=config["personal_best_weight"],
                moisture_sigma=config["moisture_sigma"],
                moisture_base=config["moisture_base"],
                evaporation_rate=config["evaporation_rate"],
                split_ratio=config["split_ratio"],
                child_offset=config["child_offset"],
                initial_root_count=config["initial_root_count"],
                seed=seed,
            )

            best_pos, best_fit = algo.run(max_iterations=max_iterations)
            fitnesses.append(float(best_fit))
            positions.append(best_pos)

        mean_fit, std_fit = mean_std(fitnesses)
        best_run_idx = int(np.argmin(fitnesses))
        best_fit = float(fitnesses[best_run_idx])
        best_pos = positions[best_run_idx]

        score = stable_normalized_score(mean_fit, optimum=problem.optimum)
        aggregate_scores.append(score)

        per_problem[problem.name] = {
            "computed_step_size": problem_step_size,
            "mean_fitness": mean_fit,
            "std_fitness": std_fit,
            "best_fitness": best_fit,
            "best_position": best_pos.tolist(),
            "all_fitnesses": [float(x) for x in fitnesses],
            "score": score,
        }

    overall_score = float(np.mean(aggregate_scores))

    config_out = {k: serialize_value(v) for k, v in config.items()}
    config_out["step_scale"] = step_scale

    return {
        "config": config_out,
        "overall_score": overall_score,
        "per_problem": per_problem,
    }


# ------------------------------------------------------------
# Logging
# ------------------------------------------------------------

def flatten_result_for_csv(result: Dict[str, Any]) -> Dict[str, Any]:
    row: Dict[str, Any] = {}

    row["overall_score"] = result["overall_score"]

    for k, v in result["config"].items():
        if isinstance(v, (list, dict)):
            row[f"param_{k}"] = json.dumps(v)
        else:
            row[f"param_{k}"] = v

    for problem_name, metrics in result["per_problem"].items():
        prefix = problem_name.lower()
        row[f"{prefix}_computed_step_size"] = metrics["computed_step_size"]
        row[f"{prefix}_mean_fitness"] = metrics["mean_fitness"]
        row[f"{prefix}_std_fitness"] = metrics["std_fitness"]
        row[f"{prefix}_best_fitness"] = metrics["best_fitness"]
        row[f"{prefix}_best_position"] = json.dumps(metrics["best_position"])
        row[f"{prefix}_all_fitnesses"] = json.dumps(metrics["all_fitnesses"])
        row[f"{prefix}_score"] = metrics["score"]

    return row


def write_results_csv(results: List[Dict[str, Any]], filepath: str) -> None:
    rows = [flatten_result_for_csv(r) for r in results]
    if not rows:
        return

    fieldnames: List[str] = []
    seen = set()
    for row in rows:
        for key in row.keys():
            if key not in seen:
                seen.add(key)
                fieldnames.append(key)

    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def write_best_config_json(result: Dict[str, Any], filepath: str) -> None:
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)


# ------------------------------------------------------------
# Random search
# ------------------------------------------------------------

def random_grid_search_with_logs(
    search_space: Dict[str, Sequence[Any]],
    problems: Sequence[BenchmarkProblem],
    n_samples: int = 40,
    seeds_per_config: Sequence[int] = (0, 1, 2),
    max_iterations: int = 1200,
    random_state: int = 123,
    top_k: int = 10,
    step_scale: float = 0.05,
    all_results_csv: str = "all_results.csv",
    top_results_csv: str = "top_results.csv",
    best_config_json: str = "best_config.json",
) -> List[Dict[str, Any]]:
    rng = random.Random(random_state)
    seen = set()
    all_results: List[Dict[str, Any]] = []

    max_unique = 1
    for values in search_space.values():
        max_unique *= len(values)

    actual_samples = min(n_samples, max_unique)

    while len(all_results) < actual_samples:
        config = sample_param_config(rng, search_space)

        key = tuple(
            (k, tuple(v) if isinstance(v, (list, tuple, np.ndarray)) else v)
            for k, v in sorted(config.items(), key=lambda item: item[0])
        )
        if key in seen:
            continue
        seen.add(key)

        result = evaluate_config(
            config=config,
            problems=problems,
            seeds=seeds_per_config,
            max_iterations=max_iterations,
            step_scale=step_scale,
        )
        all_results.append(result)

        write_results_csv(all_results, all_results_csv)

        print(
            f"[{len(all_results)}/{actual_samples}] "
            f"score={result['overall_score']:.6f}"
        )

    all_results.sort(key=lambda r: r["overall_score"])
    top_results = all_results[:top_k]

    write_results_csv(all_results, all_results_csv)
    write_results_csv(top_results, top_results_csv)
    write_best_config_json(top_results[0], best_config_json)

    return top_results


# ------------------------------------------------------------
# Search space
# ------------------------------------------------------------

def make_search_space(dimension: int = 3) -> Dict[str, Sequence[Any]]:
    zero_gravity = tuple([0.0] * dimension)

    return {
        "max_root_count": [40, 60, 80],
        "initial_energy": [4.0, 6.0, 8.0, 10.0],
        "step_cost": [0.01, 0.02, 0.03, 0.05],
        "splitting_threshold": [4.0, 5.0, 6.0, 8.0],
        "max_children": [1, 2, 3],
        "alpha": [2.0, 4.0, 8.0, 12.0],
        "deposit_scale": [0.5, 1.0, 2.0, 4.0, 8.0],
        "depletion_scale": [0.005, 0.01, 0.03, 0.05, 0.08],
        "weights": [
            (1.0, 0.0, 0.10, 0.20),
            (1.4, 0.0, 0.15, 0.30),
            (1.8, 0.0, 0.25, 0.45),
            (2.2, 0.0, 0.10, 0.60),
            (2.5, 0.0, 0.05, 0.80),
        ],
        "repulsion_radius": [0.4, 0.8, 1.2, 1.8],
        "gravity_vector": [zero_gravity],
        "personal_best_weight": [0.4, 0.8, 1.2, 1.8, 2.5],
        "moisture_sigma": [0.4, 0.7, 0.9, 1.2, 1.8],
        "moisture_base": [0.001, 0.01, 0.05, 0.1],
        "evaporation_rate": [0.001, 0.005, 0.01, 0.03, 0.05],
        "split_ratio": [0.25, 0.45, 0.60],
        "child_offset": [0.02, 0.06, 0.12, 0.20],
        "initial_root_count": [4, 8, 12],
    }


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

if __name__ == "__main__":
    DIMENSION = 3
    MAX_ITERATIONS = 100
    N_SAMPLES = 5
    SEEDS_PER_CONFIG = [0, 1, 2, 3, 4]
    STEP_SCALE = 0.05

    problems = make_problems(dimension=DIMENSION)
    search_space = make_search_space(dimension=DIMENSION)

    top_results = random_grid_search_with_logs(
        search_space=search_space,
        problems=problems,
        n_samples=N_SAMPLES,
        seeds_per_config=SEEDS_PER_CONFIG,
        max_iterations=MAX_ITERATIONS,
        random_state=42,
        top_k=5,
        step_scale=STEP_SCALE,
        all_results_csv="all_results.csv",
        top_results_csv="top_results.csv",
        best_config_json="best_config.json",
    )

    print("\nTop 5 configurations:")
    for i, result in enumerate(top_results, start=1):
        print(f"Rank #{i}: score={result['overall_score']:.6f}")
        print(result["config"])
        for problem_name, metrics in result["per_problem"].items():
            print(f"  {problem_name} step_size = {metrics['computed_step_size']:.6f}")

# Benchmarking & Analysis

In this section the algorithm is tested on Rastrigin, Ackley, Schwefel and Rosenbrock function.

The result shows that the algorithm demonstrably works, shows meaningful convergence on multiple standard benchmarks, and performs especially well on Ackley and Rosenbrock, but still struggles on highly multimodal landscapes like Rastrigin and Schwefel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

In [ ]:
# Benchmark functions

def rastrigin(x: np.ndarray) -> float:
        x = np.asarray(x, dtype=float)
        n = x.size
        return float(10.0 * n + np.sum(x**2 - 10.0 * np.cos(2.0 * np.pi * x)))

def ackley(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    n = x.size
    sum_sq = np.sum(x**2)
    sum_cos = np.sum(np.cos(2.0 * np.pi * x))
    term1 = -20.0 * np.exp(-0.2 * np.sqrt(sum_sq / n))
    term2 = -np.exp(sum_cos / n)
    return float(term1 + term2 + 20.0 + np.e)

def schwefel(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    n = x.size
    return float(418.9829 * n - np.sum(x * np.sin(np.sqrt(np.abs(x)))))

def rosenbrock(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    return float(np.sum(100.0 * (x[1:] - x[:-1] ** 2) ** 2 + (1.0 - x[:-1]) ** 2))

def compute_step_size(bounds, scale=0.05):
        bounds_arr = np.asarray(bounds, dtype=float)
        widths = bounds_arr[:, 1] - bounds_arr[:, 0]
        return scale * float(np.mean(widths))

def func_dict(name):
    if name == "rastrigin":
        return rastrigin
    elif name == "ackley":
        return ackley
    elif name == "schwefel":
        return schwefel
    elif name == "rosenbrock":
        return rosenbrock
    else:
        raise ValueError("Unknown function")

In [ ]:
# Parameters

dimension = 5
population = 60
iterations = 200
runs = 10

test_cases = {
    "rastrigin": {
        "func": func_dict("rastrigin"),
        "bounds": (-5.12, 5.12),
    },
    "ackley": {
        "func": func_dict("ackley"),
        "bounds": (-32.768, 32.768),
    },
    "schwefel": {
        "func": func_dict("schwefel"),
        "bounds": (-500.0, 500.0),
    },
    "rosenbrock": {
        "func": func_dict("rosenbrock"),
        "bounds": (-2.048, 2.048),
    },
}

In [ ]:
def normalize_bounds(bounds, dimension):
    """
    Returns:
        scalar_bounds: (low, high)
        vector_bounds: [(low, high), ...] length=dimension
        widths: np.ndarray of per-dimension widths
    """
    # case: (-5.12, 5.12)
    if isinstance(bounds[0], (int, float)):
        low, high = bounds
        scalar_bounds = (float(low), float(high))
        vector_bounds = [scalar_bounds] * dimension
    else:
        # case: [(-5.12, 5.12), ..., (-5.12, 5.12)]
        vector_bounds = [tuple(map(float, b)) for b in bounds]
        scalar_bounds = vector_bounds[0]

    widths = np.array([hi - lo for lo, hi in vector_bounds], dtype=float)
    return scalar_bounds, vector_bounds, widths


def run_benchmark(
    optimizer_cls,
    test_cases,
    optimizer_params=None,
    dimension=3,
    iterations=200,
    runs=10,
    seed_each_run=True,
    plot=True,
    plant_step_scale=0.05,
    plant_child_offset_scale=0.5,
):
    if optimizer_params is None:
        optimizer_params = {}

    results = {}

    if plot:
        plt.figure(figsize=(12, 6))

    if isinstance(test_cases, dict):
        cases_iter = [
            (name, case["func"], case["bounds"])
            for name, case in test_cases.items()
        ]
    else:
        cases_iter = test_cases

    for func_name, objective_function, bounds in cases_iter:
        print(f"\nRunning {func_name} benchmark ({runs} runs)")

        scalar_bounds, vector_bounds, widths = normalize_bounds(bounds, dimension)
        mean_width = float(np.mean(widths))

        all_histories = []
        final_scores = []
        final_positions = []
        times = []

        for run in range(runs):
            if seed_each_run:
                np.random.seed(run)

            start_time = time.time()
            params = dict(optimizer_params)

            # Common parameters
            params["objective_function"] = objective_function

            cls_name = optimizer_cls.__name__

            if cls_name == "PlantAlgorithm":
                # PlantAlgorithm expects per-dimension bounds
                params["bounds"] = vector_bounds

                # Auto-scale bound-dependent parameters if not explicitly given
                params["step_size"] = params.get("step_size", plant_step_scale * mean_width)
                params["child_offset"] = params.get(
                    "child_offset",
                    plant_child_offset_scale * params["step_size"]
                )

                # Optional: adapt gravity vector to dimension if not provided
                params["gravity_vector"] = np.asarray(
                    params.get("gravity_vector", np.zeros(dimension)),
                    dtype=float
                )

                params["seed"] = run

                optimizer = optimizer_cls(**params)
                best_position, best_fitness = optimizer.run(max_iterations=iterations)

            elif cls_name == "ArtificialBeeColony":
                # ABC usually expects scalar bounds + num_dimensions
                params["bounds"] = scalar_bounds
                params.setdefault("num_dimensions", dimension)
                params.setdefault("max_iterations", iterations)

                optimizer = optimizer_cls(**params)
                best_position, best_fitness = optimizer.run(max_iterations=iterations)

            elif cls_name == "Swarm":
                # Adjust this branch if your Swarm constructor differs
                params["bounds"] = scalar_bounds
                params.setdefault("dimension", dimension)
                params.setdefault("pop_size", params.pop("pop_size", 60))

                optimizer = optimizer_cls(**params)
                best_position, best_fitness = optimizer.run(
                    objective_function=objective_function,
                    iterations=iterations
                )

            else:
                # Generic fallback
                params["bounds"] = scalar_bounds
                params.setdefault("num_dimensions", dimension)

                optimizer = optimizer_cls(**params)
                best_position, best_fitness = optimizer.run(max_iterations=iterations)

            history = np.asarray(getattr(optimizer, "history", [best_fitness]), dtype=float)
            elapsed = time.time() - start_time

            all_histories.append(history)
            final_scores.append(best_fitness)
            final_positions.append(best_position)
            times.append(elapsed)

        max_len = max(len(h) for h in all_histories)
        padded_histories = []

        for h in all_histories:
            if len(h) < max_len:
                h = np.pad(h, (0, max_len - len(h)), mode="edge")
            padded_histories.append(h)

        all_histories = np.array(padded_histories)
        final_scores = np.array(final_scores, dtype=float)
        final_positions = np.array(final_positions, dtype=float)
        times = np.array(times, dtype=float)

        mean_curve = np.mean(all_histories, axis=0)
        mean_score = np.mean(final_scores)
        std_score = np.std(final_scores)
        mean_time = np.mean(times)
        mean_position = np.mean(final_positions, axis=0)

        print(f"\n{func_name} Summary")
        print(f"  Mean Best Score   : {mean_score:.6e} ± {std_score:.2e}")
        print(f"  Mean Best Position: {np.round(mean_position, 4)}")
        print(f"  Mean Time         : {mean_time:.4f} sec")

        results[func_name] = {
            "mean_curve": mean_curve,
            "mean_score": mean_score,
            "std_score": std_score,
            "mean_time": mean_time,
            "mean_position": mean_position,
            "all_scores": final_scores,
            "all_positions": final_positions,
            "all_times": times,
            "all_histories": all_histories,
        }

        if plot:
            plt.plot(np.maximum(mean_curve, 1e-12), label=f"{func_name} (mean)")

    if plot:
        plt.title(f"Benchmark Convergence Curve (Dim = {dimension})")
        plt.xlabel("Iterations")
        plt.ylabel("Best Fitness (Log Scale)")
        plt.yscale("log")
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.legend()
        plt.tight_layout()
        plt.show()

    return results

In [ ]:
from RootAlgorithm import PlantAlgorithm

results_plant = run_benchmark(
    optimizer_cls=PlantAlgorithm,
    test_cases=test_cases,
    optimizer_params={
        "max_root_count": population,
        "initial_energy": 6.0,
        "step_cost": 0.03,
        "splitting_threshold": 8.0,
        "max_children": 2,
        "alpha": 12.0,
        "deposit_scale": 2.5,
        "depletion_scale": 0.02,
        "weights": (1.8, 0.0, 0.4, 0.3),
        "repulsion_radius": 1.8,
        "gravity_vector": np.zeros(dimension),
        "personal_best_weight": 0.8,
        "moisture_sigma": 0.4,
        "moisture_base": 0.01,
        "evaporation_rate": 0.005,
        "split_ratio": 0.6,
        "initial_root_count": 12,
    },
    dimension=dimension,
    iterations=iterations,
    runs=runs,
    plant_step_scale=0.05,          # step_size = 5% of average search width
    plant_child_offset_scale=0.5,   # child_offset = 0.5 * step_size
)

# PSO and ABC

Comparing the algorithm against ABC and PSO the first thing we notice is that computational demand for the plant algorithm is quite high. Running the program with the same functions and iterations resulted in much longer wait time than ABC and PSO combined. Both of which ran in a few seconds while our algorithm took over a minute.

The convergence results indicate distinct performance characteristics among the three algorithms across the benchmark functions. The proposed plant-inspired algorithm demonstrates stable and consistent convergence behavior, particularly in the early iterations, often outperforming PSO in terms of initial convergence speed. This suggests that the combination of gradient-following (hydrotropism) and inter-agent repulsion provides an effective balance of exploration and exploitation in the early search phase. However, the algorithm tends to prematurely plateau at higher fitness values, failing to refine solutions to near-optimal levels (i.e., close to zero) on most functions. In contrast, PSO shows moderate convergence but also exhibits stagnation, particularly on multimodal functions like Rastrigin and Schwefel. The Artificial Bee Colony (ABC) algorithm achieves the best overall performance, consistently converging to significantly lower fitness values and demonstrating strong global search capability. This superior performance can be attributed to ABC’s dynamic role-based search mechanism, which maintains diversity while intensifying exploitation around promising regions.

Overall, while the plant algorithm is competitive in terms of robustness and early-stage convergence, its limited fine-tuning ability (weak exploitation phase) restricts its final solution quality, indicating a need for enhanced local search mechanisms or adaptive parameter control.

In [ ]:
from ABCAlgorithm import ArtificialBeeColony

results_abc = run_benchmark(
    optimizer_cls=ArtificialBeeColony,
    test_cases=test_cases,
    optimizer_params={
        "num_food_sources": population,
        "limit": (population * dimension) // 2,
    },
    dimension=dimension,
    iterations=iterations,
    runs=runs,
)

In [ ]:

from PSOAlgorithm import Swarm

results = run_benchmark(
    optimizer_cls=Swarm,
    test_cases=test_cases,
    dimension=dimension,
    iterations=iterations,
    runs=runs,
    optimizer_params={
        "pop_size": population,
        "dimension": dimension
    }
)